# FTG Noise Analysis

The functions here are for the purpose of checking for high noise in FTG data.

Here we demonstrate their use on the Vinton Dome airborne gravity gradiometer survey data.

Ensure you have run the `Prepare_XYZ` notebook first so that the Vinton Dome data are prepared for review.

___

Import the required modules, and set the path to the geowhizz files.

In [ ]:
from pathlib import Path
import pegasusQC as qc

In [ ]:
vintonHDF_file = Path(r'./VintonData/VintonDome.hdf5')

In [ ]:
if not vintonHDF_file.exists():
    %run ./Prepare_VintonDomeData.ipynb

___

The **in-line sum** of FTG data is defined as the sum of the three in-line component data divided by $\sqrt{3}$, after bandpass filtering from $0.03\,Hz$ to $0.1\,Hz$, or similar frequencies.

By construction, the in-line sum should ideally be zero and higher values are indicative of the noise in the data.

The `ilsNoiseAnalysis` function reports flight-lines with in-line sum greater than the given threshold (the `noiseSpec`) and plots the in-line sum for each survey line against the turbulence for that line. Higher turbulence generally results in higher noise and the plot allows one to decide on a reasonable turbulence limit for the data. In addition, flight-lines with an in-line sum that is off the general trend (high in-line sum at low turbulence) should be questioned even if they meet specification.

The Vinton Dome data used in testing did not include a turbulence channel but did have a vertical velocity channel. This is supplied to `ilsNoiseAnalysis` and differenced to form a vertical acceleration channel as a proxy for turbulence.

In this example, we get very low noise estimates because we are inputting (despite the channel names) final transformed data. The check should be applied to raw data.

In [ ]:
qc.ilsNoiseAnalysis(vintonHDF_file, 'Inline1_raw', 'Inline2_raw', 'Inline3_raw', noiseSpec=17.0, vertdispl='altitude')

___

Any rectification process can down-convert high frequency vibration into the signal band of a sensor. Gravity gradiometers have an intrinsic rectification process via their sensitivity to products of rotational velocity so it is useful to check for excess high frequency signal since it may lead to error in the final data.

The function **`checkHighFreq`** checks for unusually high amplitude, high frequency signal in the raw gradient channels. The input gradient channels are high-pass filtered and a rolling standard deviation is used to find periods of higher amplitudes. High turbulence is sometimes associated with high frequency noise so the turbulence is also plotted for comparison.

Experience with FTG data suggests that sections of data where `checkHighFreq` finds high frequency noise above $50\,E$ are of concern and ought to be followed up with the service provider.

A plot showing where the high frequency noise occurs along line is shown for all lines that have excess high frequency signal. Despits the names of the input channels, the Vinton Dome data have been filtered and so the `noiseLimit` has been artifically set here to $14\,E$ just to get a plot for the example.

In [ ]:
qc.checkHighFreq(vintonHDF_file, noiseLimit=14,
                 channels=['Cross1_raw', 'Cross2_raw', 'Cross3_raw', 'Inline1_raw', 'Inline2_raw', 'Inline3_raw'],
                 cutoffs=[0.1, 0.48], vertdispl='altitude', verbose=False, plot_flag=True)